# Imports

In [ ]:
# =============================================================================
# Imports and environment setup
# =============================================================================
import os
import sys
import time
import logging
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Make the kernel robust if launched through VSCode/Jupyter rather than
# from an already activated conda shell.
CONDA_PREFIX = Path(sys.executable).resolve().parents[1]

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONNOUSERSITE"] = "1"

# Make conda libraries preferred inside Jupyter.
# This is important on Juno because scipy/sklearn may otherwise pick up
# /opt/ohpc/pub/apps/miniconda3/lib/libstdc++.so.6 and fail with GLIBCXX_3.4.30.
os.environ["LD_LIBRARY_PATH"] = (
    f"{CONDA_PREFIX / 'lib'}:" + os.environ.get("LD_LIBRARY_PATH", "")
)

import numpy as np
import pandas as pd
import scipy
import scipy.optimize
import anndata as ad
import scanpy as sc
import torch

# Workaround for mixed matplotlib installs where pyplot expects rcParams._get.
import matplotlib
if not hasattr(matplotlib.rcParams, "_get"):
    matplotlib.rcParams._get = matplotlib.rcParams.__getitem__
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn import metrics as sk_metrics

import INSTINCT

try:
    from IPython.display import display
except Exception:
    display = print

print("Python:", sys.executable)
print("CONDA_PREFIX:", CONDA_PREFIX)
print("INSTINCT:", INSTINCT.__file__)
print("torch:", torch.__version__, "cuda:", torch.version.cuda, "available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scipy:", scipy.__version__)
print("matplotlib:", matplotlib.__version__)


# USER CONFIGURATION — edit these to match your environment

In [ ]:
# =============================================================================
# USER CONFIGURATION
# =============================================================================

# Path to cloned INSTINCT repo
INSTINCT_ROOT = Path(r"/work/dal875013/projects/mocha/methods/INSTINCT")

# Path to the folder containing RCC_TLS_10x .h5ad files
DATA_ROOT = Path(r"/groups/qiwei/mocha/QuickSRT/RCC_TLS_10x/data")

# Output folder for benchmark-only results
# This notebook intentionally writes only:
#   1. benchmark.log
#   2. performance.parquet
#   3. predictions.parquet
OUTPUT_DIR = Path(r"/work/dal875013/projects/mocha/results/INSTINCT/rcc_tls_instinct")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METHOD_NAME = "INSTINCT"
DATASET_NAME = "RCC_TLS_10x"

# Subject-level RCC_TLS_10x grouping, based on the dataset README.
# Only subject 22 is included, as requested.
# Subject 22 has two samples: one FFPE and one frozen sample. INSTINCT is run once
# on this two-sample subject group.
RCC_TLS_SUBJECT_GROUPS = {
    "subject_22": ["GSM5924030_ffpe_c_2", "GSM5924050_frozen_c_2"],
}

# Fixed number of spatial domains for RCC_TLS_10x benchmarking.
# User-specified: use 3 clusters for subject 22.
RCC_TLS_NUM_CLUSTERS = 3

# ----- SUBSET SELECTOR -----------------------------------------------------
# Only subject 22 is included. Keep this as None to run subject_22.
# Equivalently, set SUBJECT_GROUPS_TO_RUN = "subject_22".
SUBJECT_GROUPS_TO_RUN = None

# Number of repeated runs per subject group.
NUM_RUNS = 1

# Keep the output directory benchmark-only.
# Any old intermediate files from earlier versions of this notebook will be removed
# before each benchmark run.
BENCHMARK_OUTPUT_FILES = {"benchmark.log", "performance.parquet", "predictions.parquet"}
CLEAN_OUTPUT_DIR_BEFORE_RUN = True

# INSTINCT tutorial-style choices.
N_TOP_GENES = 5000
PCA_N_COMPONENTS = 100
GMM_COVARIANCE_TYPE = "tied"

# If None, auto-detect ground-truth column from adata.obs.
GT_LABEL_COL = None

# Random seed used by PCA/GMM.
BASE_SEED = 1234

# =============================================================================
# Helpers
# =============================================================================
def setup_logging(output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    log_path = output_dir / "benchmark.log"

    logger = logging.getLogger()
    logger.setLevel(logging.INFO)

    # Avoid duplicated handlers when rerunning notebook cells.
    for h in list(logger.handlers):
        logger.removeHandler(h)

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    fh = logging.FileHandler(log_path, mode="w")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    logging.info(f"Logging to {log_path}")


def clean_output_dir_for_benchmark_only(output_dir: Path):
    """
    Keep OUTPUT_DIR limited to the three benchmark outputs only.

    This removes legacy/intermediate files produced by older notebook versions,
    such as PCA .npy files, latent .csv files, noise embedding .csv files,
    group-level .h5ad files, or saved figures.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    if not CLEAN_OUTPUT_DIR_BEFORE_RUN:
        return

    removed = []
    for p in output_dir.iterdir():
        if p.is_file() and p.name not in BENCHMARK_OUTPUT_FILES:
            p.unlink()
            removed.append(p.name)

    if removed:
        preview = ", ".join(removed[:10])
        suffix = " ..." if len(removed) > 10 else ""
        print(f"[CLEANUP] Removed {len(removed)} non-benchmark file(s): {preview}{suffix}")



def resolve_subject_groups(selector):
    """Resolve SUBJECT_GROUPS_TO_RUN into dict: subject_name -> sample_ids."""
    if selector is None or selector == "all":
        return dict(RCC_TLS_SUBJECT_GROUPS)
    if isinstance(selector, str):
        return {selector: RCC_TLS_SUBJECT_GROUPS[selector]}
    if isinstance(selector, (list, tuple)):
        return {name: RCC_TLS_SUBJECT_GROUPS[name] for name in selector}
    raise ValueError(f"Unrecognized SUBJECT_GROUPS_TO_RUN value: {selector!r}")


def detect_gt_column(adata) -> str:
    """Auto-detect which obs column holds the ground-truth tissue/domain annotation."""
    if GT_LABEL_COL is not None:
        if GT_LABEL_COL not in adata.obs.columns:
            raise KeyError(
                f"GT_LABEL_COL={GT_LABEL_COL!r} not in adata.obs. "
                f"Available columns: {list(adata.obs.columns)}"
            )
        return GT_LABEL_COL

    candidates = [
        "Manual_Annotation", "ManualAnnotation",
        "annotation", "Annotation", "annotations", "Annotations",
        "ground_truth", "Ground Truth", "GroundTruth", "true_label",
        "label", "Label", "domain", "Domain", "spatial_domain",
        "tissue", "tissue_type", "histology", "region", "Region",
        "cell_type", "celltype", "cluster", "z",
    ]
    for c in candidates:
        if c in adata.obs.columns:
            return c

    raise KeyError(
        f"Could not auto-detect ground-truth column. "
        f"adata.obs columns: {list(adata.obs.columns)}. "
        f"Please set GT_LABEL_COL manually."
    )


def find_h5ad_file(sample_id: str) -> Path:
    """Find the .h5ad file for a sample, allowing common QuickSRT filename patterns."""
    candidates = [
        DATA_ROOT / f"SCE_{sample_id}.h5ad",
        DATA_ROOT / f"{sample_id}.h5ad",
        DATA_ROOT / f"adata_{sample_id}.h5ad",
    ]
    for p in candidates:
        if p.exists():
            return p

    matches = sorted(DATA_ROOT.glob(f"*{sample_id}*.h5ad"))
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        logging.warning(f"Multiple .h5ad matches for {sample_id}; using first: {matches[0]}")
        return matches[0]

    raise FileNotFoundError(
        f"Could not find .h5ad for sample {sample_id!r} in {DATA_ROOT}. "
        f"Tried SCE_{sample_id}.h5ad, {sample_id}.h5ad, adata_{sample_id}.h5ad, and glob *{sample_id}*.h5ad."
    )


def ensure_spatial_obsm(adata, sample_id: str):
    """Create adata.obsm['spatial'] from common coordinate locations if needed."""
    if "spatial" in adata.obsm:
        return
    if "S" in adata.obsm:
        adata.obsm["spatial"] = np.asarray(adata.obsm["S"])
        logging.info(f"    [DIAG] {sample_id}: aliased obsm['S'] -> obsm['spatial']")
        return

    coord_pairs = [
        ("x", "y"),
        ("X", "Y"),
        ("spatial_x", "spatial_y"),
        ("array_col", "array_row"),
        ("pxl_col_in_fullres", "pxl_row_in_fullres"),
        ("imagecol", "imagerow"),
    ]
    for x_col, y_col in coord_pairs:
        if x_col in adata.obs.columns and y_col in adata.obs.columns:
            adata.obsm["spatial"] = adata.obs[[x_col, y_col]].to_numpy(dtype=float)
            logging.info(
                f"    [DIAG] {sample_id}: created obsm['spatial'] from obs[{x_col!r}, {y_col!r}]"
            )
            return

    raise KeyError(
        f"No spatial coordinates found for sample {sample_id}. "
        f"Available obsm keys: {list(adata.obsm.keys())}; obs columns: {list(adata.obs.columns)}"
    )


def to_dense(x):
    """Convert sparse/dense matrix to a dense numpy array."""
    if hasattr(x, "toarray"):
        return x.toarray()
    return np.asarray(x)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Data loading  (adapted for RCC_TLS_10x .h5ad input)

In [ ]:
# =============================================================================
# Data loading  (adapted for RCC_TLS_10x .h5ad input)
# =============================================================================
def load_one_rcc_tls_sample_for_instinct(sample_id: str):
    """
    Load one RCC_TLS_10x .h5ad sample and convert it into the format used by
    the INSTINCT SRT workflow.

    Returns:
        adata  : AnnData for one sample
        gt_col : original ground-truth obs column
    """
    h5ad_path = find_h5ad_file(sample_id)
    logging.info(f"    [LOAD] Reading {h5ad_path}")

    adata = sc.read_h5ad(h5ad_path)
    adata.var_names_make_unique()

    logging.info(
        f"    [DIAG] {sample_id}: n_obs={adata.n_obs}, n_vars={adata.n_vars}"
    )
    logging.info(f"    [DIAG] {sample_id}: obs columns={list(adata.obs.columns)}")
    logging.info(f"    [DIAG] {sample_id}: obsm keys={list(adata.obsm.keys())}")

    # ---- Ground-truth label column ----
    gt_col = detect_gt_column(adata)
    adata.obs["Manual_Annotation"] = adata.obs[gt_col].astype(object)
    adata.obs["sample"] = sample_id
    adata.obs["original_spot_id"] = adata.obs_names.astype(str)

    n_labeled = int(adata.obs["Manual_Annotation"].notna().sum())
    logging.info(
        f"    [DIAG] {sample_id}: using GT column '{gt_col}' "
        f"(labeled spots: {n_labeled}/{adata.n_obs})"
    )
    logging.info(
        f"    [DIAG] {sample_id}: label counts="
        f"{adata.obs['Manual_Annotation'].astype(str).value_counts().head(30).to_dict()}"
    )

    # ---- Spatial coordinates ----
    ensure_spatial_obsm(adata, sample_id)

    # Make obs_names unique across samples, while preserving original_spot_id.
    adata.obs_names = [f"{x}_{sample_id}" for x in adata.obs_names.astype(str)]

    return adata, gt_col


def load_rcc_tls_subject_group_for_instinct(subject_name: str, sample_ids: list):
    """
    Load two RCC_TLS_10x samples for one subject group.

    Returns:
        rna_list     : list of AnnData, one per sample
        adata_concat : concatenated AnnData before INSTINCT preprocessing
        gt_cols      : dict sample_id -> original GT column
    """
    logging.info(f"  [SUBJECT] Loading {subject_name}: {sample_ids}")

    rna_list = []
    gt_cols = {}

    for sample_id in sample_ids:
        adata, gt_col = load_one_rcc_tls_sample_for_instinct(sample_id)
        rna_list.append(adata)
        gt_cols[sample_id] = gt_col

    adata_concat = ad.concat(
        rna_list,
        label="slice_name",
        keys=sample_ids,
        index_unique=None,
    )

    logging.info(
        f"  [DIAG] Combined {subject_name}: n_obs={adata_concat.n_obs}, "
        f"n_vars={adata_concat.n_vars}, samples={sample_ids}"
    )

    return rna_list, adata_concat, gt_cols


# Metrics and subject-level INSTINCT training

In [ ]:
# =============================================================================
# Metrics
# =============================================================================
def map_gmm_clusters_to_true_labels(true_labels, cluster_labels):
    """
    Match unsupervised GMM clusters to ground-truth labels with Hungarian matching.

    Returns:
        matched_pred_labels : array of label names, same length as cluster_labels
        cluster_to_label    : dict raw_cluster -> matched true label
    """
    true = pd.Series(true_labels).astype(object)
    pred = pd.Series(cluster_labels).astype(object)

    valid = true.notna() & pred.notna()
    true_valid = true[valid].astype(str).to_numpy()
    pred_valid = pred[valid].astype(str).to_numpy()

    true_cats = np.array(sorted(pd.unique(true_valid)))
    pred_cats = np.array(sorted(pd.unique(pred_valid)))

    if len(true_cats) == 0 or len(pred_cats) == 0:
        return np.array([f"Cluster_{x}" for x in pred.astype(str)]), {}

    # Cost matrix = negative overlap, because linear_sum_assignment minimizes.
    overlap = np.zeros((len(true_cats), len(pred_cats)), dtype=int)
    for i, t in enumerate(true_cats):
        for j, p in enumerate(pred_cats):
            overlap[i, j] = np.sum((true_valid == t) & (pred_valid == p))

    row_ind, col_ind = scipy.optimize.linear_sum_assignment(-overlap)
    cluster_to_label = {pred_cats[j]: true_cats[i] for i, j in zip(row_ind, col_ind)}

    matched = np.array([
        cluster_to_label.get(str(x), f"Cluster_{x}") if pd.notna(x) else np.nan
        for x in pred
    ], dtype=object)

    return matched, cluster_to_label


def compute_cluster_metrics(y_true, y_pred):
    """
    Compute clustering metrics used in the STG3Net/SpaCross-style summary.

    ACC is a summary score defined here as mean(NMI, HOM, COM).
    DIS is kept as NaN to preserve the same output schema; INSTINCT's official
    tutorial reports ARI/AMI/NMI/FMI/Completeness/Homogeneity instead of DIS.
    """
    y_true = pd.Series(y_true).astype(object)
    y_pred = pd.Series(y_pred).astype(object)

    valid = y_true.notna() & y_pred.notna()
    y_true = y_true[valid].astype(str).to_numpy()
    y_pred = y_pred[valid].astype(str).to_numpy()

    ARI = float(sk_metrics.adjusted_rand_score(y_true, y_pred))
    AMI = float(sk_metrics.adjusted_mutual_info_score(y_true, y_pred))
    NMI = float(sk_metrics.normalized_mutual_info_score(y_true, y_pred))
    FMI = float(sk_metrics.fowlkes_mallows_score(y_true, y_pred))
    HOM = float(sk_metrics.homogeneity_score(y_true, y_pred))
    COM = float(sk_metrics.completeness_score(y_true, y_pred))
    ACC = float(np.nanmean([NMI, HOM, COM]))
    DIS = np.nan

    return ARI, NMI, ACC, DIS, AMI, FMI, HOM, COM


# =============================================================================
# Train one subject group and return sample-level metrics
# =============================================================================
def train_one_subject_group(
    subject_name: str,
    sample_ids: list,
    rna_list: list,
    adata_concat,
    device,
    run_seed: int = 0,
):
    """
    Train INSTINCT once on one RCC_TLS subject group, then compute metrics separately
    for each sample.

    Returns:
        metrics_list : list of per-sample metric dicts
        pred_df      : DataFrame with SpaCross/STG3Net-compatible prediction columns

    Output policy:
        This function does not write PCA matrices, latent embeddings, noise
        embeddings, or group-level h5ad intermediates. The notebook only writes
        benchmark.log, performance.parquet, and predictions.parquet in run_benchmark().
    """
    num_clusters = int(RCC_TLS_NUM_CLUSTERS)

    logging.info(
        f"  [TRAIN] Starting INSTINCT | subject={subject_name} | "
        f"samples={sample_ids} | k_clusters={num_clusters} | "
        f"device={device} | run_seed={run_seed}"
    )

    t0 = time.time()

    np.random.seed(BASE_SEED + run_seed)
    torch.manual_seed(BASE_SEED + run_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(BASE_SEED + run_seed)

    # -------------------------------------------------------------------------
    # INSTINCT SRT tutorial-style sequence:
    #   1. preprocess_SRT(..., n_top_genes=5000)
    #   2. PCA to 100 dimensions
    #   3. create_neighbor_graph(...)
    #   4. INSTINCT_Model(...).train(...)
    #   5. eval(rna_list)
    #   6. GMM clustering on result.obsm['INSTINCT_latent']
    # -------------------------------------------------------------------------
    logging.info("  [PREPROCESS] INSTINCT.preprocess_SRT")
    rna_list, adata_concat = INSTINCT.preprocess_SRT(
        rna_list,
        adata_concat,
        n_top_genes=N_TOP_GENES,
    )

    logging.info(
        f"  [DIAG] after preprocess: n_obs={adata_concat.n_obs}, n_vars={adata_concat.n_vars}"
    )

    n_components = min(PCA_N_COMPONENTS, adata_concat.n_obs - 1, adata_concat.n_vars)
    logging.info(f"  [PCA] Reducing feature dimension to {n_components}")
    pca = PCA(n_components=n_components, random_state=BASE_SEED + run_seed)
    input_matrix = pca.fit_transform(to_dense(adata_concat.X))
    adata_concat.obsm["X_pca"] = input_matrix

    # Benchmark-only mode: keep the PCA matrix in memory only.

    logging.info("  [GRAPH] INSTINCT.create_neighbor_graph")
    INSTINCT.create_neighbor_graph(rna_list, adata_concat)

    model = INSTINCT.INSTINCT_Model(rna_list, adata_concat, device=device)

    logging.info("  [MODEL] Training INSTINCT")
    model.train(report_loss=True, report_interval=100)

    logging.info("  [MODEL] Evaluating INSTINCT")
    model.eval(rna_list)

    result = ad.concat(
        rna_list,
        label="slice_name",
        keys=sample_ids,
        index_unique=None,
    )

    latent = np.asarray(result.obsm["INSTINCT_latent"])
    # Benchmark-only mode: do not write latent/noise embeddings to disk.

    logging.info("  [CLUSTER] GaussianMixture on INSTINCT_latent")
    gm = GaussianMixture(
        n_components=num_clusters,
        covariance_type=GMM_COVARIANCE_TYPE,
        random_state=BASE_SEED + run_seed,
    )
    raw_clusters = gm.fit_predict(latent)
    result.obs["gm_cluster"] = pd.Series(raw_clusters, index=result.obs.index, dtype="category")

    matched_pred, cluster_to_label = map_gmm_clusters_to_true_labels(
        result.obs["Manual_Annotation"],
        result.obs["gm_cluster"],
    )
    result.obs["pred_label"] = matched_pred

    logging.info(f"  [DIAG] {subject_name}: cluster_to_label={cluster_to_label}")
    logging.info(
        f"  [DIAG] {subject_name}: raw cluster sizes="
        f"{pd.Series(raw_clusters).astype(str).value_counts().to_dict()}"
    )

    # Benchmark-only mode: do not write group-level h5ad intermediates.

    elapsed = time.time() - t0
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

    metrics_list = []
    pred_frames = []

    # Sample-level metrics and predictions.
    for sample_id in sample_ids:
        mask = result.obs["sample"].astype(str).to_numpy() == str(sample_id)
        sample_adata = result[mask].copy()
        labeled_sample = sample_adata[~pd.isnull(sample_adata.obs["Manual_Annotation"])].copy()

        ARI, NMI, ACC, DIS, AMI, FMI, HOM, COM = compute_cluster_metrics(
            labeled_sample.obs["Manual_Annotation"],
            labeled_sample.obs["pred_label"],
        )

        metrics = {
            "method": METHOD_NAME,
            "dataset": DATASET_NAME,
            "sample": sample_id,
            "run_seed": run_seed,
            "ARI": float(ARI),
            "NMI": float(NMI),
            "ACC": float(ACC),
            "DIS": float(DIS) if not pd.isna(DIS) else np.nan,
            # Allocate group runtime across samples so sum(running_time_sec)
            # approximates the subject-level training time.
            "running_time_sec": round(elapsed / max(1, len(sample_ids)), 2),
            "gpu_model": gpu_name,
            "training_unit": subject_name,
            "training_samples": ",".join(sample_ids),
            "group_running_time_sec": round(elapsed, 2),
            "num_clusters": int(num_clusters),
            "num_spots": int(sample_adata.n_obs),
            "num_labeled_spots": int(labeled_sample.n_obs),
            "num_training_spots": int(result.n_obs),
            "num_genes_after_hvg": int(adata_concat.n_vars),
            "clust_method": "GaussianMixture",
            "device": str(device),
            "AMI": float(AMI),
            "FMI": float(FMI),
            "HOM": float(HOM),
            "COM": float(COM),
        }
        metrics_list.append(metrics)

        logging.info(
            f"  [METRICS] {subject_name}/{sample_id}: "
            f"ARI={ARI:.4f}  NMI={NMI:.4f}  ACC={ACC:.4f}  "
            f"AMI={AMI:.4f}  FMI={FMI:.4f}  "
            f"allocated_time={elapsed / len(sample_ids):.1f}s  "
            f"group_time={elapsed:.1f}s  gpu={gpu_name}"
        )

        spot_ids = (
            sample_adata.obs["original_spot_id"].astype(str).to_numpy()
            if "original_spot_id" in sample_adata.obs.columns
            else sample_adata.obs_names.astype(str).to_numpy()
        )

        pred_frames.append(
            pd.DataFrame(
                {
                    "method": METHOD_NAME,
                    "dataset": DATASET_NAME,
                    "sample": sample_id,
                    "run_seed": run_seed,
                    "spot_id": spot_ids,
                    "true_label": sample_adata.obs["Manual_Annotation"].astype(object).to_numpy(),
                    "pred_label": sample_adata.obs["pred_label"].astype(object).to_numpy(),
                    "pred_cluster": sample_adata.obs["gm_cluster"].astype(str).to_numpy(),
                }
            )
        )

    pred_df = pd.concat(pred_frames, ignore_index=True)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics_list, pred_df


# Main benchmark loop

In [ ]:
# =============================================================================
# Main benchmark loop
# =============================================================================
def run_benchmark():
    clean_output_dir_for_benchmark_only(OUTPUT_DIR)
    setup_logging(OUTPUT_DIR)

    logging.info("=" * 72)
    logging.info("INSTINCT RCC_TLS_10x Subject-Level Benchmarking Framework")
    logging.info("=" * 72)

    if not INSTINCT_ROOT.exists():
        raise FileNotFoundError(
            f"INSTINCT_ROOT not found: {INSTINCT_ROOT.resolve()}\n"
            f"Clone it with: git clone https://github.com/yyLIU12138/INSTINCT.git"
        )

    if str(INSTINCT_ROOT.resolve()) not in sys.path:
        sys.path.insert(0, str(INSTINCT_ROOT.resolve()))

    if not DATA_ROOT.exists():
        raise FileNotFoundError(f"DATA_ROOT not found: {DATA_ROOT.resolve()}")

    device = get_device()

    logging.info(f"[PATHS] INSTINCT_ROOT = {INSTINCT_ROOT.resolve()}")
    logging.info(f"[PATHS] DATA_ROOT     = {DATA_ROOT.resolve()}")
    logging.info(f"[PATHS] OUTPUT_DIR    = {OUTPUT_DIR.resolve()}")
    logging.info(f"[ENV] Python          = {sys.executable}")
    logging.info(f"[ENV] INSTINCT        = {INSTINCT.__file__}")
    logging.info(f"[ENV] torch           = {torch.__version__}")
    logging.info(f"[ENV] cuda_available  = {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        logging.info(f"[ENV] gpu             = {torch.cuda.get_device_name(0)}")

    subject_groups = resolve_subject_groups(SUBJECT_GROUPS_TO_RUN)
    logging.info(f"[SUBJECTS] Will process {len(subject_groups)} subject group(s): {subject_groups}")
    logging.info(
        f"[SETTINGS] num_runs={NUM_RUNS}  n_top_genes={N_TOP_GENES}  "
        f"pca={PCA_N_COMPONENTS}  fixed_clusters={RCC_TLS_NUM_CLUSTERS}"
    )
    logging.info("[MODE] subject-level multi-sample training; sample-level metrics/output")
    logging.info("[OUTPUT] benchmark-only: benchmark.log, performance.parquet, predictions.parquet")

    all_metrics = []
    all_predictions = []
    failed = []

    for i, (subject_name, sample_ids) in enumerate(subject_groups.items(), start=1):
        logging.info("")
        logging.info("-" * 72)
        logging.info(f"[{i}/{len(subject_groups)}] Subject group {subject_name}: {sample_ids}")
        logging.info("-" * 72)

        try:
            for run in range(NUM_RUNS):
                logging.info(f"  -> Run {run + 1}/{NUM_RUNS}")

                # Reload the data for each run because INSTINCT preprocessing/model training
                # modifies rna_list and adata_concat in place.
                rna_list, adata_concat, gt_cols = load_rcc_tls_subject_group_for_instinct(
                    subject_name=subject_name,
                    sample_ids=sample_ids,
                )

                metrics_list, pred_df = train_one_subject_group(
                    subject_name=subject_name,
                    sample_ids=sample_ids,
                    rna_list=rna_list,
                    adata_concat=adata_concat,
                    device=device,
                    run_seed=run,
                )

                for row in metrics_list:
                    row["gt_col"] = gt_cols.get(row["sample"], GT_LABEL_COL)
                all_metrics.extend(metrics_list)
                all_predictions.append(pred_df)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        except Exception as e:
            logging.exception(f"[FAILED] Subject group {subject_name}: {e}")
            failed.append({"training_unit": subject_name, "error": str(e)})
            continue

    if not all_metrics:
        logging.error("No successful runs — nothing to save.")
        return

    predictions = pd.concat(all_predictions, ignore_index=True)
    performance = pd.DataFrame(all_metrics)

    perf_lead = [
        "method", "dataset", "sample", "run_seed",
        "ARI", "NMI", "ACC", "DIS",
        "running_time_sec", "gpu_model",
        "training_unit", "training_samples", "group_running_time_sec",
    ]
    performance = performance[
        perf_lead + [c for c in performance.columns if c not in perf_lead]
    ]

    # Keep prediction file mostly SpaCross/STG3Net-compatible, with raw pred_cluster added.
    pred_lead = ["method", "dataset", "sample", "run_seed", "spot_id", "true_label", "pred_label", "pred_cluster"]
    predictions = predictions[pred_lead]

    pred_path = OUTPUT_DIR / "predictions.parquet"
    perf_path = OUTPUT_DIR / "performance.parquet"
    predictions.to_parquet(pred_path, index=False)
    performance.to_parquet(perf_path, index=False)

    logging.info("")
    logging.info("=" * 72)
    logging.info("SAVED")
    logging.info("=" * 72)
    logging.info(f"  benchmark.log       -> {OUTPUT_DIR / 'benchmark.log'}")
    logging.info(f"  predictions.parquet -> {pred_path}  ({len(predictions):,} rows)")
    logging.info(f"  performance.parquet -> {perf_path}  ({len(performance):,} rows)")

    logging.info("")
    logging.info("Per-sample metrics:")
    show_cols = [
        "training_unit", "sample", "ARI", "NMI", "ACC", "DIS",
        "AMI", "FMI", "HOM", "COM",
        "running_time_sec", "group_running_time_sec",
        "num_clusters", "num_labeled_spots",
    ]
    show_cols = [c for c in show_cols if c in performance.columns]
    logging.info("\n" + performance[show_cols].to_string(index=False))

    logging.info("")
    logging.info(
        f"Overall mean ARI: {performance['ARI'].mean():.4f}  "
        f"NMI: {performance['NMI'].mean():.4f}  "
        f"ACC: {performance['ACC'].mean():.4f}"
    )
    logging.info(f"GPU model(s) used: {sorted(performance['gpu_model'].unique().tolist())}")
    logging.info(
        "Total subject-level running time: "
        f"{performance[['training_unit', 'run_seed', 'group_running_time_sec']].drop_duplicates()['group_running_time_sec'].sum():.1f}s"
    )

    if failed:
        logging.warning(f"\n{len(failed)} subject group(s) failed:")
        for f_ in failed:
            logging.warning(f"  - {f_['training_unit']}: {f_['error']}")

    logging.info("\nBenchmarking complete.")


# Execute the benchmark when this cell is run.
run_benchmark()


# Prediction results visualization (reads predictions.parquet only)

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib
if not hasattr(matplotlib.rcParams, "_get"):
    matplotlib.rcParams._get = matplotlib.rcParams.__getitem__
import matplotlib.pyplot as plt
from pathlib import Path

DATA_ROOT  = Path(r"/groups/qiwei/mocha/QuickSRT/RCC_TLS_10x/data")
OUTPUT_DIR = Path(r"/work/dal875013/projects/mocha/results/INSTINCT/rcc_tls_instinct")

SAMPLE_ID = "GSM5924030_ffpe_c_2"

# Helper copied here so this visualization cell can run independently.
def find_h5ad_file_for_plot(sample_id: str) -> Path:
    candidates = [
        DATA_ROOT / f"SCE_{sample_id}.h5ad",
        DATA_ROOT / f"{sample_id}.h5ad",
        DATA_ROOT / f"adata_{sample_id}.h5ad",
    ]
    for p in candidates:
        if p.exists():
            return p
    matches = sorted(DATA_ROOT.glob(f"*{sample_id}*.h5ad"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"No .h5ad found for {sample_id} in {DATA_ROOT}")

predictions = pd.read_parquet(OUTPUT_DIR / "predictions.parquet")
pred_slice = predictions[predictions["sample"] == SAMPLE_ID].set_index("spot_id")
adata = sc.read_h5ad(find_h5ad_file_for_plot(SAMPLE_ID))

adata.obs["true_label"] = pred_slice["true_label"].reindex(adata.obs_names).values
adata.obs["pred_label"] = pred_slice["pred_label"].reindex(adata.obs_names).values
adata.obs["pred_cluster"] = pred_slice["pred_cluster"].reindex(adata.obs_names).values

if "spatial" not in adata.obsm:
    if "S" in adata.obsm:
        adata.obsm["spatial"] = np.asarray(adata.obsm["S"])
    elif {"x", "y"}.issubset(adata.obs.columns):
        adata.obsm["spatial"] = adata.obs[["x", "y"]].to_numpy(dtype=float)
    elif {"array_col", "array_row"}.issubset(adata.obs.columns):
        adata.obsm["spatial"] = adata.obs[["array_col", "array_row"]].to_numpy(dtype=float)
    else:
        raise KeyError(f"No spatial coordinates found. obsm={list(adata.obsm.keys())}, obs={list(adata.obs.columns)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc.pl.embedding(
    adata, basis="spatial", color="true_label",
    ax=axes[0], show=False, title=f"{SAMPLE_ID} — Ground Truth", s=20,
)
sc.pl.embedding(
    adata, basis="spatial", color="pred_label",
    ax=axes[1], show=False, title=f"{SAMPLE_ID} — INSTINCT Prediction (K=3)", s=20,
)
axes[0].invert_yaxis()
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()


# Subject-level 2-sample visualization (reads predictions.parquet only)

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib
if not hasattr(matplotlib.rcParams, "_get"):
    matplotlib.rcParams._get = matplotlib.rcParams.__getitem__
import matplotlib.pyplot as plt
from pathlib import Path

DATA_ROOT  = Path(r"/groups/qiwei/mocha/QuickSRT/RCC_TLS_10x/data")
OUTPUT_DIR = Path(r"/work/dal875013/projects/mocha/results/INSTINCT/rcc_tls_instinct")

RCC_TLS_SUBJECT_GROUPS_FOR_PLOT = {
    "subject_22": ["GSM5924030_ffpe_c_2", "GSM5924050_frozen_c_2"],
}

SUBJECT_TO_PLOT = "subject_22"
SAMPLES_TO_PLOT = RCC_TLS_SUBJECT_GROUPS_FOR_PLOT[SUBJECT_TO_PLOT]

# Helper copied here so this visualization cell can run independently.
def find_h5ad_file_for_plot(sample_id: str) -> Path:
    candidates = [
        DATA_ROOT / f"SCE_{sample_id}.h5ad",
        DATA_ROOT / f"{sample_id}.h5ad",
        DATA_ROOT / f"adata_{sample_id}.h5ad",
    ]
    for p in candidates:
        if p.exists():
            return p
    matches = sorted(DATA_ROOT.glob(f"*{sample_id}*.h5ad"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"No .h5ad found for {sample_id} in {DATA_ROOT}")

predictions = pd.read_parquet(OUTPUT_DIR / "predictions.parquet")

fig, axes = plt.subplots(2, len(SAMPLES_TO_PLOT), figsize=(5 * len(SAMPLES_TO_PLOT), 8))
if len(SAMPLES_TO_PLOT) == 1:
    axes = np.array(axes).reshape(2, 1)
fig.suptitle(f"INSTINCT RCC_TLS_10x {SUBJECT_TO_PLOT}: Ground Truth vs Cluster Results (K=3)", fontsize=16)

for j, sample_id in enumerate(SAMPLES_TO_PLOT):
    pred_slice = predictions[predictions["sample"] == sample_id].set_index("spot_id")
    adata = sc.read_h5ad(find_h5ad_file_for_plot(sample_id))

    adata.obs["true_label"] = pred_slice["true_label"].reindex(adata.obs_names).values
    adata.obs["pred_label"] = pred_slice["pred_label"].reindex(adata.obs_names).values
    adata.obs["pred_cluster"] = pred_slice["pred_cluster"].reindex(adata.obs_names).values

    if "spatial" not in adata.obsm:
        if "S" in adata.obsm:
            adata.obsm["spatial"] = np.asarray(adata.obsm["S"])
        elif {"x", "y"}.issubset(adata.obs.columns):
            adata.obsm["spatial"] = adata.obs[["x", "y"]].to_numpy(dtype=float)
        elif {"array_col", "array_row"}.issubset(adata.obs.columns):
            adata.obsm["spatial"] = adata.obs[["array_col", "array_row"]].to_numpy(dtype=float)
        else:
            raise KeyError(f"No spatial coordinates found for {sample_id}")

    sc.pl.embedding(
        adata, basis="spatial", color="true_label",
        ax=axes[0, j], show=False, title=f"{sample_id} — Ground Truth", s=20,
        legend_loc=None,
    )
    sc.pl.embedding(
        adata, basis="spatial", color="pred_label",
        ax=axes[1, j], show=False, title=f"{sample_id} — INSTINCT", s=20,
        legend_loc=None,
    )
    axes[0, j].invert_yaxis()
    axes[1, j].invert_yaxis()

plt.tight_layout()
plt.show()


# Running results summary

In [ ]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(r"/work/dal875013/projects/mocha/results/INSTINCT/rcc_tls_instinct")
perf_path = OUTPUT_DIR / "performance.parquet"

if not perf_path.exists():
    raise FileNotFoundError(f"Cannot find {perf_path}")

performance = pd.read_parquet(perf_path)
performance = performance.sort_values(["training_unit", "sample", "run_seed"]).reset_index(drop=True)

display_cols = [
    "training_unit", "sample", "run_seed",
    "ARI", "NMI", "ACC", "DIS",
    "AMI", "FMI", "HOM", "COM",
    "num_clusters", "num_spots", "num_labeled_spots",
    "running_time_sec", "group_running_time_sec",
    "clust_method", "gpu_model",
]
display_cols = [c for c in display_cols if c in performance.columns]

print("===== All INSTINCT subject-level RCC_TLS_10x results =====")
display(performance[display_cols])

print("\n===== Summary =====")
summary_cols = [c for c in ["ARI", "NMI", "ACC", "DIS", "AMI", "FMI", "HOM", "COM", "running_time_sec"] if c in performance.columns]
summary = performance[summary_cols].describe()
display(summary)

print("\n===== Mean metrics =====")
print(f"Mean ARI: {performance['ARI'].mean():.4f}")
print(f"Mean NMI: {performance['NMI'].mean():.4f}")
print(f"Mean ACC: {performance['ACC'].mean():.4f}")
if "AMI" in performance.columns:
    print(f"Mean AMI: {performance['AMI'].mean():.4f}")
if "FMI" in performance.columns:
    print(f"Mean FMI: {performance['FMI'].mean():.4f}")
if "DIS" in performance.columns:
    print(f"Mean DIS: {performance['DIS'].mean():.4f}")

print("\n===== Subject-level runtime =====")
runtime = performance[["training_unit", "run_seed", "group_running_time_sec"]].drop_duplicates()
display(runtime)
print(f"Total subject-level runtime: {runtime['group_running_time_sec'].sum():.2f} sec")

print("\n===== Best / worst ARI =====")
best = performance.loc[performance["ARI"].idxmax()]
worst = performance.loc[performance["ARI"].idxmin()]

print(f"Best ARI : {best['sample']} ({best['training_unit']})  ARI={best['ARI']:.4f}")
print(f"Worst ARI: {worst['sample']} ({worst['training_unit']})  ARI={worst['ARI']:.4f}")
